# Extension: Calibrated Abstention

**Falsifiable hypothesis (do not edit after the first result run).** A single jointly fine-tuned multilingual QA model produces a *language-dependent* distribution of the no-answer margin $s_{\text{null}} - s_{\text{best}}$. Therefore per-language validation-tuned no-answer thresholds will raise answerability macro-F1 relative to one pooled global threshold, mainly by increasing unanswerable-class recall for the language with the shifted score distribution, while leaving exact match on answerable examples unchanged.

**Secondary hypothesis.** Post-hoc temperature scaling of the two-way {no-answer, best span} softmax will reduce expected calibration error but will *not* change answerability macro-F1 at an optimised threshold, because a strictly monotone rescaling of the margin cannot change the set of achievable dichotomies.

**Output of this notebook:** printed tables plus CSV files in `results/`. You copy the numbers you need into the LaTeX tables by hand.

Everything is tuned on the **validation split only**. The challenge set is never touched here.


## 1. Setup
Registers the environment, fixes seeds, makes `src/` importable. In Colab, upload `src/abstention.py` and `src/results_io.py` next to the notebook.


In [ ]:
import os, sys, json, platform, time, random
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip -q install 'transformers>=4.40' 'datasets>=2.19' scikit-learn matplotlib pyarrow

ROOT = Path('/content') if IN_COLAB else Path('.').resolve()
for p in (ROOT, ROOT/'src', ROOT.parent/'src'):
    if (p/'abstention.py').exists():
        sys.path.insert(0, str(p)); break
else:
    raise SystemExit('Put abstention.py and results_io.py in ./src first.')

import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch, transformers, datasets
import abstention as ab
import results_io as rio

SEED = 13
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

ENV = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'datasets': datasets.__version__,
    'numpy': np.__version__,
    'hardware': torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor(),
    'seed': SEED,
}
print(json.dumps(ENV, indent=2))


## 2. Configuration
`MODEL_NAME` must be the **same frozen Week 40/41 checkpoint** the core pipeline uses, or the extension is not comparable. Point it at your fine-tuned directory or the shared Hub id. The `deepset` fallback exists only so the notebook runs end to end before your checkpoint is wired in — output from it is a smoke test, not a result.


In [ ]:
CFG = dict(
    dataset='coastalcph/tydi_xor_rc',
    langs=('ar','ko','te'),
    split='validation',
    model_name=os.environ.get('QA_MODEL', 'deepset/xlm-roberta-base-squad2'),
    max_length=384, stride=128, max_answer_len=30,
    max_examples_per_lang=None,
    target_abstention_recall=0.90,
    cache=Path('cache/val_scores.parquet'),
    figures=Path('figures'), results=Path('results'),
)
CFG['cache'].parent.mkdir(parents=True, exist_ok=True)
for k in ('figures','results'): CFG[k].mkdir(parents=True, exist_ok=True)
print('model:', CFG['model_name'])


## 3. Validation data
Loads the three mandatory languages and normalises the answerability column (`answertable` in the released dataset).


In [ ]:
from datasets import load_dataset

df_raw = load_dataset(CFG['dataset'], split=CFG['split']).to_pandas()
ans_col = 'answertable' if 'answertable' in df_raw.columns else 'answerable'
df_raw['answerable'] = df_raw[ans_col].astype(bool).astype(int)
df_raw = df_raw[df_raw['lang'].isin(CFG['langs'])].reset_index(drop=True)
df_raw['id'] = [f"{r.lang}-{i}" for i, r in df_raw.iterrows()]

if CFG['max_examples_per_lang']:
    df_raw = (df_raw.groupby('lang', group_keys=False)
                     .apply(lambda d: d.sample(min(len(d), CFG['max_examples_per_lang']),
                                               random_state=SEED)).reset_index(drop=True))

counts = df_raw.groupby('lang')['answerable'].agg(n='size', answerable_rate='mean').round(3)
display(counts)
print('total', len(df_raw))
counts.to_csv(CFG['results']/'val_counts.csv')


## 4. Frozen QA model
Loaded once and kept frozen for every condition. This is the only GPU-heavy step and its output is cached.


In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
model = AutoModelForQuestionAnswering.from_pretrained(CFG['model_name'])
model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'{n_params/1e6:.0f}M parameters on {device}')


## 5. Score the validation split (cached)
For each example we store $s_{\text{null}}$ (min over windows of the CLS-span score) and $s_{\text{best}}$ (max over valid in-context spans), plus the predicted span and its EM/F1 against gold. Every condition downstream is a function of these columns only, so the analysis reruns on CPU in seconds.


In [ ]:
if CFG['cache'].exists():
    scores = pd.read_parquet(CFG['cache'])
    minutes = float('nan')
    print('loaded cache', scores.shape)
else:
    t0 = time.time()
    scores = ab.score_examples(df_raw.to_dict('records'), model, tokenizer,
                               ab.ScoringConfig(model_name=CFG['model_name'],
                                                max_length=CFG['max_length'],
                                                stride=CFG['stride'],
                                                max_answer_len=CFG['max_answer_len'],
                                                batch_size=CFG['batch_size']),
                               device=device)
    minutes = (time.time()-t0)/60
    scores.to_parquet(CFG['cache'])
    print(f'scored {len(scores)} examples in {minutes:.1f} min')

display(scores.groupby('lang')[['s_null','s_best','margin','em']].mean().round(3))
score_minutes = round(minutes, 1) if minutes == minutes else 'cached'


## 6. Sanity checks before any claiming
Catches the three ways this analysis silently breaks: empty spans, NaN margins, a leaked feature.


In [ ]:
assert set(scores['lang']) <= set(CFG['langs']), 'unexpected language'
assert scores['margin'].notna().all(), 'NaN margins'
assert (scores['s_null'] > -1e8).all() and (scores['s_best'] > -1e8).all(), 'empty window'
print(scores.groupby('lang')['margin'].describe(percentiles=[.25,.5,.75]).round(3))
print('empty predictions:', int((scores['pred_span']=='').sum()),
      '| answerable:', int(scores['answerable'].sum()))

fig, ax = plt.subplots(figsize=(5,3))
scores.boxplot(column='margin', by='lang', ax=ax)
ax.set_ylabel('s_null - s_best'); ax.set_title('No-answer margin by language')
fig.suptitle(''); fig.tight_layout()
fig.savefig(CFG['figures']/'fig_margin_by_lang.png', dpi=200)


## 7. Conditions A–F under the shared protocol
| | Rule | Tuned parameters |
|---|---|---|
| A | pooled threshold maximising answerability macro-F1 (core baseline) | 1 |
| B | one threshold per language | 3 |
| C | temperature scaling of the two-way softmax, then one pooled threshold | 2 |
| D | threshold fixed by a constraint: abstention recall ≥ 0.90, per language | 3 |
| E | logistic regression on post-hoc features, out-of-fold, then a pooled threshold | 6 |
| F | fixed 0.5 cut on the calibrated probability (no validation labels used) | 0 |


In [ ]:
res = ab.run_conditions(scores, target_recall=CFG['target_abstention_recall'], seed=SEED)
main = pd.DataFrame(res['conditions']).T[rio.MAIN_COLS].round(3)
main.index = [rio.COND_LABELS.get(k, k) for k in main.index]
display(main)
display(pd.DataFrame(res['thresholds']).T)


## 8. Per-language breakdown
The brief requires results separately per language with the number of evaluated examples.


In [ ]:
pl = pd.concat([d.assign(condition=k) for k,d in res['by_language'].items()], ignore_index=True)
pl['condition'] = pl['condition'].map(rio.COND_LABELS)
pl[pl['condition'].str.startswith(('A.','B.','D.'))]\
  .pivot(index='lang', columns='condition',
         values=['macro_f1','coverage_answerable','abstention_recall']).round(3)


## 9. Paired per-example transitions
The part that carries the argument: who moved and in which direction. `rescued_abstention` = unanswerable example A answered and B correctly refused; `newly_wrong_abstention` = answerable example B wrongly refused.


In [ ]:
display(res['transitions_AB'])
display(res['transitions_AD'])
print({k: (round(v,4) if isinstance(v,float) else v) for k,v in res['bootstrap_AB'].items()})
print('AUC of the feature combiner (E):', round(res['extra']['logreg_auc'], 3))
if res['extra']['logreg_auc'] > 0.98:
    print('WARNING: AUC near 1.0 -> a feature is leaking the label.')
    print('span_len must come from the model prediction, never from the gold answer.')


## 10. Calibration: does rescaling change the decision?
Compare ECE and macro-F1 after the threshold is retuned. If temperature scaling only moves the first, the finding is clean: calibration fixes *reported confidence*, per-language thresholds fix *decisions*.


In [ ]:
print('T =', round(res['extra']['temperature'],3),
      '| ECE raw', round(res['extra']['ece_raw'],4),
      '-> calibrated', round(res['extra']['ece_temperature'],4))
print('macro-F1 A vs C:', round(res['conditions']['A_classic']['macro_f1'],4),
      round(res['conditions']['C_temperature']['macro_f1'],4))

fig, ax = plt.subplots(figsize=(3.6,3.6))
ax.plot([0,1],[0,1],'--',color='grey',lw=1)
ax.plot(res['reliability_raw']['confidence'], res['reliability_raw']['accuracy'],'o-',label='raw')
ax.plot(res['reliability_cal']['confidence'], res['reliability_cal']['accuracy'],'s-',label='temperature scaled')
ax.set_xlabel('predicted P(answerable)'); ax.set_ylabel('observed answerable fraction')
ax.legend(frameon=False); fig.tight_layout()
fig.savefig(CFG['figures']/'fig_reliability.png', dpi=200)
display(res['reliability_raw'].round(3)); display(res['reliability_cal'].round(3))


## 11. Risk–coverage
Threshold-free view of ranking quality. A and B share the same curve: per-language thresholds move the *cut point*, not the ranking. That is why the per-language offset is the intervention that matters.


In [ ]:
fig, ax = plt.subplots(figsize=(4,3))
ax.plot(res['curve_A']['coverage'], res['curve_A']['risk'], label=f"A (AURC={res['extra']['aurc_A']:.3f})")
ax.set_xlabel('coverage (fraction answered)'); ax.set_ylabel('risk (1 - EM on answered)')
ax.legend(frameon=False); fig.tight_layout()
fig.savefig(CFG['figures']/'fig_risk_coverage.png', dpi=200)
print('AURC per language:', {k: round(v,3) for k,v in res['extra']['aurc_by_lang'].items()})


## 12. Error inspection
Concrete cases to quote in the report's error-analysis paragraph.


In [ ]:
p = res['predictions']
rescued = p[(p['pred_A_classic']==1) & (p['pred_B_per_language']==0) & (p['answerable']==0)]
broken  = p[(p['pred_A_classic']==0) & (p['pred_B_per_language']==1) & (p['answerable']==1)]
cols = ['lang','question','margin','pred_span','gold_answer']
print('--- rescued abstentions'); display(rescued[cols].head(8))
print('--- broken abstentions');  display(broken[cols].head(8))


## 13. Save results to CSV
Writes the CSV/JSON files and prints every table you need. Copy the numbers you want into the LaTeX tabulars by hand; nothing is generated for you.


In [ ]:
meta = dict(model_name=CFG['model_name'], seed=SEED, hardware=ENV['hardware'],
            n_val=int(len(scores)), trainable_params=f'{n_params/1e6:.0f}M',
            score_time_min=score_minutes, torch=ENV['torch'],
            transformers=ENV['transformers'])
out = rio.save_results(res, CFG['results'], meta=meta)
rio.print_tables(res)
print('\nfiles written to', str(out.resolve()))
for f in sorted(out.iterdir()):
    print('  ', f.name, f'{f.stat().st_size/1024:.0f} KB')


## 14. What goes in the report
- **Hypothesis** — the block at the top, unchanged, with direction and metric named before the run.
- **Design** — the A–F table; state that all rules share one score, one metric implementation, one example set and one frozen model.
- **Result** — `metrics_by_condition.csv` for the main table, `metrics_by_language.csv` for the per-language table, macro-F1 primary, EM/F1 secondary, `n` per language stated.
- **Analysis** — the C-vs-A equivalence is a finding, not a null result: it separates calibrating reported confidence from fixing the abstention decision rule.
- **Error analysis** — `transitions_AB.csv` counts plus one or two quoted questions per language from cell 12.
- **Limitations** — three thresholds tuned on a small per-language validation subset (overfitting risk, state the size), one frozen checkpoint so no retraining variance, ECE computed for the answerability event rather than answer correctness, and condition F using no validation labels at all.
